In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

TASK1

In [2]:
start_date = datetime(2024, 1, 1)
end_date = datetime(2025, 1, 1)
date_range = (end_date - start_date).days
orders=[{
        "order_id": i+1,
        "customer_id":np.random.randint(1,100),
        "product_name": np.random.choice(["Laptop", "Phone", "Tablet", "Headphones", "Camera"]),
        "quantity": np.random.randint(1, 10),
        "unit_price": round(random.uniform(5, 500), 2),
        "order_date":(start_date + timedelta(days=np.random.randint(0, date_range))).strftime("%Y-%m-%d"),
        "shipping_country": np.random.choice(["Germany","Korea","Kenya","Canada","Peru","Japan","USA","Azerbaijan","Turkey","UK"])
    }
for i in range(200)]
for a in orders:
    print(a)

{'order_id': 1, 'customer_id': 82, 'product_name': np.str_('Laptop'), 'quantity': 6, 'unit_price': 118.63, 'order_date': '2024-11-26', 'shipping_country': np.str_('USA')}
{'order_id': 2, 'customer_id': 47, 'product_name': np.str_('Camera'), 'quantity': 4, 'unit_price': 179.55, 'order_date': '2024-09-19', 'shipping_country': np.str_('Azerbaijan')}
{'order_id': 3, 'customer_id': 65, 'product_name': np.str_('Tablet'), 'quantity': 5, 'unit_price': 265.12, 'order_date': '2024-12-22', 'shipping_country': np.str_('Germany')}
{'order_id': 4, 'customer_id': 95, 'product_name': np.str_('Camera'), 'quantity': 7, 'unit_price': 137.98, 'order_date': '2024-02-18', 'shipping_country': np.str_('Azerbaijan')}
{'order_id': 5, 'customer_id': 50, 'product_name': np.str_('Tablet'), 'quantity': 3, 'unit_price': 325.78, 'order_date': '2024-06-10', 'shipping_country': np.str_('Germany')}
{'order_id': 6, 'customer_id': 78, 'product_name': np.str_('Laptop'), 'quantity': 4, 'unit_price': 395.5, 'order_date': '20

In [3]:
orders

[{'order_id': 1,
  'customer_id': 82,
  'product_name': np.str_('Laptop'),
  'quantity': 6,
  'unit_price': 118.63,
  'order_date': '2024-11-26',
  'shipping_country': np.str_('USA')},
 {'order_id': 2,
  'customer_id': 47,
  'product_name': np.str_('Camera'),
  'quantity': 4,
  'unit_price': 179.55,
  'order_date': '2024-09-19',
  'shipping_country': np.str_('Azerbaijan')},
 {'order_id': 3,
  'customer_id': 65,
  'product_name': np.str_('Tablet'),
  'quantity': 5,
  'unit_price': 265.12,
  'order_date': '2024-12-22',
  'shipping_country': np.str_('Germany')},
 {'order_id': 4,
  'customer_id': 95,
  'product_name': np.str_('Camera'),
  'quantity': 7,
  'unit_price': 137.98,
  'order_date': '2024-02-18',
  'shipping_country': np.str_('Azerbaijan')},
 {'order_id': 5,
  'customer_id': 50,
  'product_name': np.str_('Tablet'),
  'quantity': 3,
  'unit_price': 325.78,
  'order_date': '2024-06-10',
  'shipping_country': np.str_('Germany')},
 {'order_id': 6,
  'customer_id': 78,
  'product_name

In [4]:
for idx in [14,25,48]:
    orders[idx]["product_name"] = "" #3record missing product name

In [5]:
for idx in [32,46]:
    orders[idx]["quantity"] = -1*orders[idx]["quantity"]#2 record with negative quantity
for idx in [71]:
    orders[idx]["unit_price"] = -1*orders[idx]["unit_price"]#1 record with negative unit_price

In [6]:
for idx in [35,49]:
    orders[idx]["order_date"] = "not-a-date"#2 record with not a date
for idx in [92]:
    orders[idx]["order_date"] = "2025-13-45"#1 record with incorrect format

In [7]:
for idx in [109,111,117]:
    orders[idx]["order_id"] = "122"#3 records with duplicate order_id values

In [8]:
for idx in [132,155]:
    orders[idx]["quantity"] = str(orders[idx]["quantity"])#2 record with string quantity
for idx in [190]:
    orders[idx]["unit_price"] = str(orders[idx]["unit_price"])#1 record with string unit_price

In [9]:
#step2
def extract(raw_records):
    valid = []
    rejected = []
    reasons = {}

    for rec in raw_records:
        try:
            # Check Missing
            if not rec.get("product_name"):
                raise ValueError("Missing product_name")
            
            # Check Numeric Types & Positive
            q = rec["quantity"]
            u = rec["unit_price"]
            if not (isinstance(q, (int, float)) and isinstance(u, (int, float))):
                raise ValueError("Quantity/Price not numeric")
            if q <= 0 or u <= 0:
                raise ValueError("Non-positive Quantity/Price")
            
            # Check Date
            datetime.strptime(str(rec["order_date"]), "%Y-%m-%d")
            
            valid.append(rec)
        except Exception as e:
            reason = str(e)
            rejected.append({"record": rec, "reason": reason})
            reasons[reason] = reasons.get(reason, 0) + 1

    print(f"Extraction Summary:\n- Valid: {len(valid)}\n- Rejected: {len(rejected)}")
    for r, count in reasons.items(): print(f"  > {r}: {count}")
    return valid, rejected

# --- Step 3: Transform ---
def transform(valid_records):
    df = pd.DataFrame(valid_records)
    
    # FIX: Ensure order_id is a consistent type to prevent PyArrow crashes
    df['order_id'] = df['order_id'].astype(str) 
    
    df['order_date'] = pd.to_datetime(df['order_date'])
    df['total_amount'] = df['quantity'] * df['unit_price']
    
    # ... rest of your transformation logic ...
    df['order_month'] = df['order_date'].dt.month
    df['order_day_of_week'] = df['order_date'].dt.day_name()
    df['shipping_country'] = df['shipping_country'].str.title()
    
    # Deduplicate now works safely on strings
    df = df.drop_duplicates(subset=['order_id'], keep='first')
    
    def categorize_amount(amt):
        if amt < 25: return "small"
        elif amt <= 100: return "medium"
        else: return "large"
        
    df['amount_category'] = df['total_amount'].apply(categorize_amount)
    
    return df

# --- Step 4 & 5: Load & Run ---
def load(df, path="orders.parquet"):
    df.to_parquet(path)
    verify_df = pd.read_parquet(path)
    return verify_df

valid, rejected = extract(orders)
clean_df = transform(valid)
loaded_df = load(clean_df)

print(f"\nPipeline Complete:\nRaw: {len(orders)} -> Valid: {len(valid)} -> Transformed: {len(clean_df)} -> Loaded: {len(loaded_df)}")

Extraction Summary:
- Valid: 188
- Rejected: 12
  > Missing product_name: 3
  > Non-positive Quantity/Price: 3
  > time data 'not-a-date' does not match format '%Y-%m-%d': 2
  > time data '2025-13-45' does not match format '%Y-%m-%d': 1
  > Quantity/Price not numeric: 3

Pipeline Complete:
Raw: 200 -> Valid: 188 -> Transformed: 185 -> Loaded: 185


In [10]:
len(loaded_df)

185

In [11]:
len(clean_df)

185

In [12]:
clean_df.dtypes

order_id                     object
customer_id                   int64
product_name                 object
quantity                      int64
unit_price                  float64
order_date           datetime64[ns]
shipping_country             object
total_amount                float64
order_month                   int32
order_day_of_week            object
amount_category              object
dtype: object

In [13]:
loaded_df.dtypes

order_id                     object
customer_id                   int64
product_name                 object
quantity                      int64
unit_price                  float64
order_date           datetime64[ns]
shipping_country             object
total_amount                float64
order_month                   int32
order_day_of_week            object
amount_category              object
dtype: object

TASK2

In [14]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

#Extract (Minimal Validation)
def extract_elt(raw_records):
    valid_raw = [r for r in raw_records if isinstance(r, dict)]
    print(f"ELT Extract: Captured {len(valid_raw)} raw records.")
    return valid_raw

#Load (Data Lake)
def load_to_lake(raw_records, path="data_lake_raw.parquet"):
    df_raw = pd.DataFrame(raw_records).astype(str) 
    df_raw.to_parquet(path, index=False)
    print(f"ELT Load: Saved messy data to {path}")
    return path

# Transform 
def transform_elt(lake_path):
    df = pd.read_parquet(lake_path)
    df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
    df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')
    df['order_date_dt'] = pd.to_datetime(df['order_date'], errors='coerce')
    clean_mask = (
        df['product_name'].notna() & (df['product_name'] != "") &
        (df['quantity'] > 0) &
        (df['unit_price'] > 0) &
        (df['order_date_dt'].notna())
    )
    
    valid_df = df[clean_mask].copy()

    valid_df['total_amount'] = valid_df['quantity'] * valid_df['unit_price']
    valid_df['order_month'] = valid_df['order_date_dt'].dt.month
    valid_df['shipping_country'] = valid_df['shipping_country'].str.title()
    valid_df = valid_df.drop_duplicates(subset=['order_id'], keep='first')
    
    print(f"ELT Transform: Cleaned data down to {len(valid_df)} records.")
    return valid_df

# Run ELT
raw_elt = extract_elt(orders)
lake_file = load_to_lake(raw_elt)
final_elt_df = transform_elt(lake_file)

ELT Extract: Captured 200 raw records.
ELT Load: Saved messy data to data_lake_raw.parquet
ELT Transform: Cleaned data down to 188 records.


In ETL model 185 records loaded into parquet.But in ELT model all 200 records are stored to data lake then transformed and reduced to 188.

In ETL model actually no problem arised.Extract phase plays a filter role and removes invalid data.But in ELT model,problems caught in Transform phase because we store raw data in data lake and after this process it go to transform phase and some problems caught here.

In adventages ETL model is more secure and requires less space because of filtering.But ELT model is more faster and flexible(reprocess raw data) and scalable(suits for more data)

I would chose ETL when data is traditional(SQL),security and storage is priority.Choose ELT model if data is Cloud Data Warehouses (BigQuery, Snowflake, Databricks).If data is high-volume, high-velocity.

TASK3

In [33]:
from datetime import datetime
from collections import defaultdict

# Shared database (simulated)
database = {"orders": [], "features": {}}

def order_service_write(order):
    database["orders"].append(order)

def feature_service_compute():
    orders = database["orders"]
    
    customer_data = defaultdict(list)
    for o in orders:
        customer_data[o["customer_id"]].append(o)

    features = {}
    
    for cid, orders in customer_data.items():
        total_orders = len(orders)
        avg_amount = sum(o["amount"] for o in orders) / total_orders
        last_order_date = max(o["date"] for o in orders)

        features[cid] = {
            "total_orders": total_orders,
            "avg_amount": avg_amount,
            "last_order_date": last_order_date
        }

    database["features"] = features

def prediction_service_read(customer_id):
    return database["features"].get(customer_id)

In [34]:
orders_store = []

def order_service_write_api(order):
    orders_store.append(order)

def order_service_get_orders():
    return orders_store

def feature_service_compute_api():
    orders = order_service_get_orders()
    
    customer_data = defaultdict(list)
    for o in orders:
        customer_data[o["customer_id"]].append(o)

    features = {}

    for cid, orders in customer_data.items():
        total_orders = len(orders)
        avg_amount = sum(o["amount"] for o in orders) / total_orders
        last_order_date = max(o["date"] for o in orders)

        features[cid] = {
            "total_orders": total_orders,
            "avg_amount": avg_amount,
            "last_order_date": last_order_date
        }

    return features

def prediction_service_request(customer_id):
    features = feature_service_compute_api()
    return features.get(customer_id)


In [35]:
class PubSubBroker:
    def __init__(self):
        self.topics = {}
        self.subscribers = {}

    def publish(self, topic, message):
        self.topics.setdefault(topic, []).append(message)
        for fn in self.subscribers.get(topic, []):
            fn(message)

    def subscribe(self, topic, handler):
        self.subscribers.setdefault(topic, []).append(handler)

    def get_latest(self, topic):
        if topic in self.topics and self.topics[topic]:
            return self.topics[topic][-1]
        return None


broker = PubSubBroker()

feature_state = {}

def order_service_publish(order):
    broker.publish("orders", order)

def feature_service_handler(order):
    cid = order["customer_id"]
    
    feature_state.setdefault(cid, [])
    feature_state[cid].append(order)

    orders = feature_state[cid]

    features = {
        "customer_id": cid,
        "total_orders": len(orders),
        "avg_amount": sum(o["amount"] for o in orders) / len(orders),
        "last_order_date": max(o["date"] for o in orders)
    }

    broker.publish("features", features)

def prediction_service_handler(feature):
    prediction_cache[feature["customer_id"]] = feature

broker.subscribe("orders", feature_service_handler)

prediction_cache = {}
broker.subscribe("features", prediction_service_handler)

In [36]:
import random

customers = ["A", "B", "C", "D"]

orders = []
for i in range(20):
    orders.append({
        "customer_id": random.choice(customers),
        "amount": random.randint(10, 200),
        "date": datetime(2026, 3, (i % 28) + 1)
    })

In [37]:
for o in orders:
    order_service_write(o)

feature_service_compute()

db_results = database["features"]

for o in orders:
    order_service_write_api(o)

api_results = feature_service_compute_api()

for o in orders:
    order_service_publish(o)

broker_results = prediction_cache

In [38]:
print("Database mode:", db_results)
print("Service mode:", api_results)
print("Broker mode:", broker_results)

assert db_results.keys() == api_results.keys() == broker_results.keys()

Database mode: {'C': {'total_orders': 5, 'avg_amount': 71.0, 'last_order_date': datetime.datetime(2026, 3, 17, 0, 0)}, 'D': {'total_orders': 4, 'avg_amount': 122.0, 'last_order_date': datetime.datetime(2026, 3, 18, 0, 0)}, 'A': {'total_orders': 8, 'avg_amount': 109.75, 'last_order_date': datetime.datetime(2026, 3, 16, 0, 0)}, 'B': {'total_orders': 3, 'avg_amount': 64.0, 'last_order_date': datetime.datetime(2026, 3, 20, 0, 0)}}
Service mode: {'C': {'total_orders': 5, 'avg_amount': 71.0, 'last_order_date': datetime.datetime(2026, 3, 17, 0, 0)}, 'D': {'total_orders': 4, 'avg_amount': 122.0, 'last_order_date': datetime.datetime(2026, 3, 18, 0, 0)}, 'A': {'total_orders': 8, 'avg_amount': 109.75, 'last_order_date': datetime.datetime(2026, 3, 16, 0, 0)}, 'B': {'total_orders': 3, 'avg_amount': 64.0, 'last_order_date': datetime.datetime(2026, 3, 20, 0, 0)}}
Broker mode: {'C': {'customer_id': 'C', 'total_orders': 5, 'avg_amount': 71.0, 'last_order_date': datetime.datetime(2026, 3, 17, 0, 0)}, 'D

In [39]:
db_results.keys() == api_results.keys() == broker_results.keys()

True

In Shared Database coupling is in highest,all services depend on the same schema and storage layer.There is low latency for reads and services read directly from storage.If the database goes down, all services stop working.

In Data passing through services coupling is moderate,services depend on APIs rather than database schema.There is higher latency than shared DB because of request/response calls.Prediction service must wait for feature service computation.If the feature service goes down, predictions cannot be generated.

In Data passing through a message broker coupling is lowest,services communicate through events rather than direct calls.Latency is usually asynchronous.If prediction service goes down, messages can remain in the broker.System is more resilient and fault tolerant.

TASK4

In [27]:
def run_batch_processing(df):
    # Daily aggregates
    batch_results = df.groupby('order_date').agg(
        total_orders=('order_id', 'count'),
        total_revenue=('total_amount', 'sum'),
        avg_order_size=('total_amount', 'mean'),
        unique_customers=('customer_id', 'nunique')
    ).reset_index()
    
    # Top product per day logic
    top_products = df.groupby(['order_date', 'product_name'])['total_amount'].sum().reset_index()
    top_products = top_products.loc[top_products.groupby('order_date')['total_amount'].idxmax()]
    
    final_batch = batch_results.merge(top_products[['order_date', 'product_name']], on='order_date')
    return final_batch

batch_df = run_batch_processing(clean_df)
print(batch_df.head())

  order_date  total_orders  total_revenue  avg_order_size  unique_customers  \
0 2024-01-02             2        1673.78         836.890                 2   
1 2024-01-03             5        4348.82         869.764                 5   
2 2024-01-10             1         305.32         305.320                 1   
3 2024-01-11             2        3732.10        1866.050                 2   
4 2024-01-12             1        1926.80        1926.800                 1   

  product_name  
0       Laptop  
1   Headphones  
2       Laptop  
3        Phone  
4       Camera  


In [28]:
class StreamProcessor:
    def __init__(self):
        self.running_revenue = 0
        self.running_orders = 0
        self.customers = set()
        self.history = [] # To maintain the last 50 orders for windowing
        
    def process(self, record):
        self.running_orders += 1
        self.running_revenue += record['total_amount']
        self.customers.add(record['customer_id'])
        self.history.append(record)
        
        # Keep window at 50
        if len(self.history) > 50:
            self.history.pop(0)
            
        if self.running_orders % 50 == 0:
            window_avg = sum(r['total_amount'] for r in self.history) / len(self.history)
            
            # Popular product in window
            prod_counts = {}
            for r in self.history:
                prod_counts[r['product_name']] = prod_counts.get(r['product_name'], 0) + 1
            top_prod = max(prod_counts, key=prod_counts.get)
            
            print(f"\n[Stream Update @ {self.running_orders} orders]")
            print(f"Running Revenue: ${self.running_revenue:,.2f}")
            print(f"Window Avg (Last 50): ${window_avg:.2f}")
            print(f"Window Top Product: {top_prod}")

streamer = StreamProcessor()
for record in clean_df.to_dict('records'):
    streamer.process(record)


[Stream Update @ 50 orders]
Running Revenue: $61,311.62
Window Avg (Last 50): $1226.23
Window Top Product: Headphones

[Stream Update @ 100 orders]
Running Revenue: $116,640.39
Window Avg (Last 50): $1106.58
Window Top Product: Laptop

[Stream Update @ 150 orders]
Running Revenue: $185,388.34
Window Avg (Last 50): $1374.96
Window Top Product: Laptop


In batch processing,total Revenue matches final sum.In stream processing also matches final sum (Cumulative).

In batch processing top product	is top per specific day.In stream processing it is top in the "sliding window".

Batch processing is mostly used in financial reporting,monthly billing.Stream processing is used in real-time dashboards,alerting on drop in sales.

Why do windowed stats differ from batch? Batch is bound by time (e.g., 00:00 to 23:59). Streaming windows are bound by count or duration (e.g., the last 50 events). If the 50th order happened yesterday and the 51st happened today, the stream window spans across the "batch" boundary.

Batch tells you the efficiency of your business (profit per day). Stream tells you the health of your system (Is the current window average crashing? Did we just stop receiving orders?).

TASK5

In [29]:
#step1
sample_cids = clean_df['customer_id'].unique()[:5]

def get_batch_features(customer_id, df):
    c_df = df[df['customer_id'] == customer_id]
    first_order = c_df['order_date'].min()
    days_since_first = (pd.Timestamp.now() - first_order).days
    
    return {
        "total_lifetime_orders": len(c_df),
        "avg_order_amount": round(c_df['total_amount'].mean(), 2),
        "days_since_first_order": days_since_first,
        "most_purchased_category": c_df['product_name'].mode()[0] if not c_df.empty else None
    }

In [30]:
def get_stream_features(customer_id, df):
    c_df = df[df['customer_id'] == customer_id].tail(10)
    last_order_time = c_df['order_date'].max()
    
    return {
        "recent_order_count": len(c_df),
        "recent_avg_amount": round(c_df['total_amount'].mean(), 2),
        "seconds_since_last_order": 3600, 
        "recent_top_category": c_df['product_name'].mode()[0] if not c_df.empty else None
    }

In [31]:
#step2
sample_customer_ids = clean_df['customer_id'].unique()[:5]

def compute_features_for_samples(customer_ids, df):
    all_combined_features = []

    for cid in customer_ids:
        customer_history = df[df['customer_id'] == cid].sort_values('order_date')
        batch_features = {
            "cust_id": cid,
            "total_lifetime_orders": len(customer_history),
            "avg_lifetime_amount": round(customer_history['total_amount'].mean(), 2),
            "most_purchased_prod": customer_history['product_name'].mode()[0]
        }
        
        recent_history = customer_history.tail(3)
        stream_features = {
            "recent_order_count": len(recent_history),
            "recent_avg_amount": round(recent_history['total_amount'].mean(), 2),
            "recent_top_product": recent_history['product_name'].mode()[0]
        }
        
        combined = {**batch_features, **stream_features}
        all_combined_features.append(combined)
    
    return all_combined_features

feature_vectors = compute_features_for_samples(sample_customer_ids, clean_df)

In [32]:
#step3
combined_features = {}

for cid in sample_cids:
    batch = get_batch_features(cid, clean_df)
    stream = get_stream_features(cid, clean_df)
    
    combined_features[cid] = {**batch, **stream}

for cid, feats in combined_features.items():
    print(f"Customer {cid}: {feats}\n")

Customer 82: {'total_lifetime_orders': 4, 'avg_order_amount': np.float64(470.34), 'days_since_first_order': 788, 'most_purchased_category': np.str_('Laptop'), 'recent_order_count': 4, 'recent_avg_amount': np.float64(470.34), 'seconds_since_last_order': 3600, 'recent_top_category': np.str_('Laptop')}

Customer 47: {'total_lifetime_orders': 3, 'avg_order_amount': np.float64(1095.29), 'days_since_first_order': 769, 'most_purchased_category': np.str_('Camera'), 'recent_order_count': 3, 'recent_avg_amount': np.float64(1095.29), 'seconds_since_last_order': 3600, 'recent_top_category': np.str_('Camera')}

Customer 65: {'total_lifetime_orders': 1, 'avg_order_amount': np.float64(1325.6), 'days_since_first_order': 441, 'most_purchased_category': np.str_('Tablet'), 'recent_order_count': 1, 'recent_avg_amount': np.float64(1325.6), 'seconds_since_last_order': 3600, 'recent_top_category': np.str_('Tablet')}

Customer 95: {'total_lifetime_orders': 1, 'avg_order_amount': np.float64(965.86), 'days_sinc

Batch processing operates on historical data in periodic jobs, producing static features for ML models. Stream processing operates on live data as it arrives, producing dynamic features that reflect the current system state. Most production ML systems need both, combining batch features (slow-changing, computed nightly) with streaming features (fast-changing, computed in real time).

The key insight is that data pipelines are the backbone of ML systems. The model gets the attention, but the pipeline that delivers data to the model determines whether the system actually works in production. Understanding ETL, dataflow modes, and processing paradigms gives you the foundation to build reliable, maintainable data systems.

Where Batch alone insufficient?
Example: Fraud Detection.
A customer might have a 5-year history of buying $20 books (Batch). If they suddenly try to buy three $2,000 laptops in 10 minutes (Stream), the Batch features will still show an "Average Order Value" that looks safe because the 5-year history dilutes the new spikes. Without the Streaming features (recent order count/frequency), the model won't realize a "high-velocity attack" is happening.

Where Stream alone insufficient?
Example: Personalized Recommendations.
If a user clicks on a "Diaper" ad once (Stream), a stream-only model might start spamming them with baby products. However, if the Batch features show this user has primarily purchased "High-End Electronics" for 3 years and has no history of baby products, the model can infer that the click was likely a one-off gift or a mistake, preventing a poor user experience.